## Extracting eye-tracking data outputs from SR

In [ ]:
# Import modules
import os
import pandas as pd
from pathlib import Path
import numpy as np
import glob
import shutil
import eyelinkio
from eyelinkio.edf.to_asc import to_asc
import tqdm as tqdm

### 1.1. Utility Functions for ID Formatting

In [ ]:
# Function to format ID
def format_id(id_str):
    parts = id_str.split('_')
    #if len(parts) != 2:
        #raise ValueError("Invalid ID format. Should be 'XXX_XX' or 'XXXX_XX' or 'XXX_X' or 'XXXX_XX'.")
    
    numeric_part = parts[0]
    letter_part = parts[1]
    
    #if not numeric_part.isdigit():
     #   print("Numeric part before '_' should consist of digits only.")    
    # Ensure numeric part is 4 digits long by padding with zeros if necessary
    padded_numeric_part = numeric_part.zfill(4)
    
    formatted_id = f"{padded_numeric_part}_{letter_part}"
    
    return formatted_id

# Fuction to fix experiment builder ID's so that they match the ET ID's
def eb_id_transform(file):
    file = file.upper()
    file = file.replace('Q', "")

    if "_" not in file:
        # Add "_" right before the first letter from the end 
        # Find the index of the first alphanumeric character
        for i, char in enumerate(file):
            if char.isalpha():
                break      
        # Insert "_" before the first alphanumeric character found
        file = file[:i] + '_' + file[i:]
        
    # add 0's to the end of the file name to make it 4 digits
    file = format_id(file)
    return file

# Function to check if file name contains task information
def has_task_info(file_name, task_info):
    return task_info in file_name

### 1.2. Load and Prepare Demographic Data

#### Export demographic data

In [ ]:
date="2026_02_12"

In [ ]:
redcap_date="2026_02_12"

In [ ]:
# Define root directory for all eye-tracking tasks 
root_dir = "/project/def-emayada/Sharing/CHUSJ-Q1K-PILOT/experimental"
output_dir= f"../"
# Print current directory 
print("This is the current directory: ", os.getcwd())
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [ ]:
for file in glob.glob(f"../../../tracking/source/demographics_redcap/{redcap_date}/*"):
    if "Demographics" in file:
        demo = pd.read_csv(file)
        print( "Demo file is: " , file)

In [ ]:
# Rename the columns we extract q1k_id and record_id
demo.rename(columns = {'q1k_proband_id_1':"proband_id",
                         'q1k_relative_idgenerated_1':"relative_id"},
                         inplace = True)

# Subset only the important columns
demo=demo[["record_id", "proband_id", "relative_id"]].drop_duplicates(subset=['record_id'])

# Combine the old IDs
demo["q1k_ID"] = demo["proband_id"].combine_first(demo["relative_id"])

# Drop those with no q1k IDsdemo=
demo=demo.loc[~demo.q1k_ID.isna()].drop(columns=["proband_id", "relative_id"])

# Create a duplicate of redcap_id
#demo["redcap_q1k_id"]=demo["q1k_ID"]


In [ ]:
demo

### 1.3. Extract and Standardize EEG Subject IDs

In [ ]:
# Create a list of all files in the EEG folders
eeg_q1k_subjects= []
truncated_eeg_q1k_subjects = []
family_id_subjects = []
sites = ["HSJ", "MHC"]
for site in sites: 
    for file in glob.glob(f"../../sourcedata/eeg/*"):
        subject_id = file.split('/')[-1]
        if "Pilots" in subject_id:
            continue
        # Skip sessions that have already been processed
       # print(subject_id)
        if "__" in subject_id:
            continue
        eeg_q1k_subjects.append(subject_id)
        if "1025" in subject_id:
            truncated_id=subject_id.split('1025')[1][1:]
        elif "MHC" in subject_id:
            truncated_id=subject_id.split('Q1K_MHC_200')[1]
        elif "3530" in subject_id:
            truncated_id=subject_id.split('3530')[1][1:]
        elif "1525" in subject_id:
            truncated_id=subject_id.split('1525')[1][1:]
        elif "HSJ" in subject_id:
            truncated_id=subject_id.split('Q1K_HSJ_100')[1]
        elif "MHC" in subject_id:
            truncated_id=subject_id.split('Q1K_MHC_200')[1]
        truncated_eeg_q1k_subjects.append(truncated_id)
       # print(truncated_id)
        length = len(subject_id)
        family_id = truncated_id.split('_')[0]
        family_id_subjects.append(family_id)


### 1.4. Create EEG-ET Lookup Table


In [ ]:
eeg_q1k_subjects_df = pd.DataFrame({'q1k_ID': eeg_q1k_subjects, 'et_ID': truncated_eeg_q1k_subjects,
                                    'family_ID': family_id_subjects, 'subject': truncated_eeg_q1k_subjects})
# Add 0s to the et_ID ID to make it 4 digits
eeg_q1k_subjects_df['et_ID'] = eeg_q1k_subjects_df['et_ID'].apply(lambda x: format_id(x))

In [ ]:
# Create bids compliant id
eeg_q1k_subjects_df['subject'] = eeg_q1k_subjects_df['et_ID']

for subject in eeg_q1k_subjects_df['subject'].unique():
    bids_id=subject.replace("_","")
    eeg_q1k_subjects_df.loc[eeg_q1k_subjects_df.subject==subject, "subject"]=bids_id
# Add BIDS folder name
eeg_q1k_subjects_df["bids_folder"] = "sub-" + eeg_q1k_subjects_df['subject']

In [ ]:
eeg_q1k_subjects_df

In [ ]:
print("There are a total of" , len(eeg_q1k_subjects_df.q1k_ID.unique()), "unique participants")

In [ ]:
# Merge REDCAP_ids
eeg_q1k_subjects_df=eeg_q1k_subjects_df.merge(demo, on="q1k_ID", how="left")

### 1.5. Identify Participants with Missing Demographics

In [ ]:
# Participants without demographics
missing_redcap=eeg_q1k_subjects_df.loc[eeg_q1k_subjects_df.record_id.isna()]
missing_redcap

In [ ]:
eeg_q1k_subjects_df.to_csv(f"../../tracking/output_dfs/id_conversion_tables/et_eeg_lookup_table.csv")

output_dir= f"../../../tracking/output_dfs/missingness/{date}/"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
    
# Save all data without EEG

missing_redcap.to_csv(os.path.join(output_dir, f"participants_eeg_no_redcap.csv"), index=False)


In [ ]:
tasks = ['GO', 'NSP', 'AS','VS','FSP','REST','RS', 'VEP', "AEP"]

### 2. Track and Export Eye-Tracking Data by Task

In [ ]:
# Track participants with missing EEG data
missing_eeg = []      # Track participants with ET data but no matching EEG data
et_subjects = []      # Store all eye-tracking subject IDs encountered
transformed_et = []   # Store transformed/standardized ET IDs
test_missing = []     # Track test/pilot data

# Define output directory for processed ET data (to be organized into BIDS format)
output_dir = f"../../derivatives/et_rt_analysis/"

# Iterate through each task
for task in tqdm.tqdm(tasks):
    if task == "NSP" or task == "RS" or task == "REST" or task == "VEP" or task == "AEP":
        # NSP does not generate .txt files and thus used only .edf 
        task_folders = glob.glob(f"/project/def-emayada/Sharing/CHUSJ-Q1K-PILOT/experimental/**/*NSP*/results/*/*.edf",  recursive=True)
    else:
        task_folders = glob.glob(f"/project/def-emayada/Sharing/CHUSJ-Q1K-PILOT/experimental/**/*{task}_pfp*.txt",  recursive=True)
            
    for task_file in tqdm.tqdm(task_folders):
        # Extract the participant folder name (original ET ID) from the file path
        et_original_id = os.path.dirname(task_file).split("/")[-1]
        et_subjects.append(et_original_id)
        participant_folder = os.path.dirname(task_file)
        # Transform the original ET ID to standardized format (e.g., 0123_P)
        transformed_id = eb_id_transform(et_original_id)
        transformed_et.append(transformed_id)
        
        # Extract the site 
        site=task_file.split("experimental/")[1].split("/")[0]

        # Check if this ET participant has corresponding EEG data
        if transformed_id in eeg_q1k_subjects_df.et_ID.values:
            new_participant = eeg_q1k_subjects_df.loc[
                eeg_q1k_subjects_df['et_ID'] == transformed_id
            ].subject.values[0]
            bids_sub_id="sub-" + new_participant
           # print(new_participant, task)
            output_folder = os.path.join(output_dir, bids_sub_id)
        else:
            
           # print("Missing EEG: ", et_original_id, task)
            # Check if the ID contains "Q" or "q" and is not from the "Pilots" task file
            if ("Q" in et_original_id or "q" in et_original_id)  and "Pilots" not in task_file:
                missingness_folder = "et_folders_no_eeg/experimental"
                missing_eeg.append([et_original_id, site, task])  # Add to the missing_eeg list
            else:
                # Otherwise, it's a test or pilot (exclude these)
                missingness_folder = "et_folders_no_eeg/tests"
               # print(task_file)

            output_folder = os.path.join(f"../../tracking/output_dfs/missingness/{date}/{missingness_folder}/", et_original_id)
            new_participant=et_original_id

        # Create the output folder if it doesn't exist
        os.makedirs(output_folder, exist_ok=True)

        # Check if files already exist for this participant and task (skip if so)
        expected_txt_file = os.path.join(output_folder, f"{new_participant}_{task}.txt")
        expected_edf_file = os.path.join(output_folder, f"{new_participant}_{task}.edf")
        
        # Skip if either expected file already exists
        if os.path.exists(expected_txt_file) or os.path.exists(expected_edf_file):
            #print(f"Skipping {new_participant} for task {task} - files already exist")
            et_subjects.append(new_participant)
            continue

        # Process files in the participant folder
        for file_name in os.listdir(participant_folder):
            source_path = os.path.join(participant_folder, file_name)
            # Handle .txt files containing task info
            if file_name.endswith('.txt') and has_task_info(file_name, task):
                new_file_name = f"{new_participant}_{task}.{file_name.split('.')[-1]}"
                destination_path = os.path.join(output_folder, new_file_name)
                shutil.copy2(source_path, destination_path)
            # Handle .edf files 
            elif file_name.endswith('.edf'):
                new_file_name = f"{new_participant}_{task}.{file_name.split('.')[-1]}"
                destination_path = os.path.join(output_folder, new_file_name)
                shutil.copy2(source_path, destination_path)
               
        # Append only once
        et_subjects.append(new_participant)

### 2.1 Convert files inside participant folders from **.edf** to **.asc**

In [ ]:
for bids_folder in glob.glob(os.path.join(output_dir, "sub-*")):
    subject_id = bids_folder.split("/")[-1].replace("sub-", "")
    
    # Get the participant's non-BIDS folder from your DataFrame
    try:
        non_bids_subfolder = eeg_q1k_subjects_df.loc[eeg_q1k_subjects_df.subject == subject_id, "q1k_ID"].values[0]
    except IndexError:
        print(f"Subject {subject_id} not found in eeg_q1k_subjects_df. Skipping.")
        continue
    
    # Construct full path for the non-BIDS folder
    non_bids_folder_path = os.path.join(
        "../../sourcedata",
        "*", "et", non_bids_subfolder
    )

    # Make sure the folder exists
    os.makedirs(non_bids_folder_path, exist_ok=True)

    for file_name in os.listdir(bids_folder):
        if not file_name.lower().endswith(".edf"):
            continue
        
        print("Processing file:", file_name)
        source = os.path.join(bids_folder, file_name)
        asc = os.path.splitext(source)[0] + ".asc"
        
        if os.path.exists(asc):
            print("Skipping existing:", asc)
        else:
            to_asc(source, asc)
            print("Converted:", source)

        # Save a copy to the non-BIDS folder
        asc_basename = os.path.basename(asc)
        destination = os.path.join(non_bids_folder_path, asc_basename)
        shutil.copy(asc, destination)
        print(f"Saved copy to non-BIDS folder: {destination}")

In [ ]:
missing_eeg_df = pd.DataFrame(missing_eeg, columns=["participants", "site", "task"])
#Collapse the task columns and drop duplicates
missing_eeg_df = (
    missing_eeg_df.groupby(["participants", "site"])   
    .agg({"task": lambda x: ", ".join(x)})  
    .reset_index()  
)

# rop duplicates based on participants
missing_eeg_df = missing_eeg_df.drop_duplicates(subset=["participants"])

missing_eeg_df.to_csv(f"../../tracking/output_dfs/missingness/{date}/participants_etfolders_but_no_eeg.csv", index=False)

In [ ]:
missing_eeg_df

### 2.3. Identify Participants with ET Data but No EEG

#### 2.3.1 Check Subjects with Folders but No EEG Data


In [ ]:
# Subjects with a folder  
subs_folder=glob.glob(f"../../sub*")
# Print the total number of subjects with folders
print("Total number of subjects with folders:", len(subs_folder))

# Subjects with EEG data
# Note: Adjust the path as necessary to match your directory structure
subs_eeg=glob.glob(f"../../sub*/*/eeg/*RS*.edf")
# Print the total number of subjects with EEG data
print("Total number of subjects with EEG data:", len(subs_eeg))

# Print the ids of subjects with folders but no EEG data
subs_folder = [Path(sub).name for sub in subs_folder]  # Normalize the paths

subs_eeg = [Path(sub).parent.parent.parent.name for sub in subs_eeg]  # Normalize the paths to get subject IDs
missing_eeg_ids = set(subs_folder) - set(subs_eeg)

# Print the total number of subjects with folders but no EEG data
print("Total number of subjects with folders but no EEG data:", len(missing_eeg_ids))
print("Subjects with folders but no EEG data:", missing_eeg_ids)


### 3. Summarize Task Completion Across ET and EEG

In [ ]:
bids_subject_per_task={}
bids_subs_per_task_overall=[]

et_subs_per_task={}
et_subs_per_task_overall=[]

task_counts = {}
sites = ["HSJ", "MHC"]
 
for task in tasks:
    count = 0
    for file in glob.glob(f"../../sub*/*/eeg/*{task}*.json", recursive=False):
        subject_id = file.split('-')[2].split('/')[0]
        bids_subs_per_task_overall.append([subject_id, task])
        count += 1
    bids_subject_per_task[task] = task_counts.get(task, 0) + count
    
for task in tasks:
    count = 0
    for file in glob.glob(f"../../derivatives/et_rt_analysis/sub*/*{task}*.txt", recursive=False):
        subject_id = file.split('-')[2].split('/')[0]
        et_subs_per_task_overall.append([subject_id, task])
        count += 1

    et_subs_per_task[task] = task_counts.get(task, 0) + count
    
    
# Replace RS with REST's value
et_subs_per_task['RS'] = et_subs_per_task['REST']

# Delete REST
del et_subs_per_task['REST']
del bids_subject_per_task['REST']


# Store lists in a dictionary
subject_dict = {
   
   "ET": list(set([sub[0] for sub in et_subs_per_task_overall])),
   "EEG": list(set([sub[0] for sub in bids_subs_per_task_overall]))

 }

# Print completion counts
for key, value in subject_dict.items():
    print(f"Total number completed {key}: {len(value)}")

# Print task completion counts
for task, count in task_counts.items():
    print(f"Total number completed {task}: {count}")
    

In [ ]:
# Print total with EEG BIDS subjects
print("Total number of BIDS subjects with EEG data per task")
bids_subject_per_task

In [ ]:
# Print total with ET data per task subjects
print("Total number of subjects with ET data per task")
et_subs_per_task

In [ ]:
# Create a DataFrame
eeg_df = pd.DataFrame(bids_subs_per_task_overall, columns=["bids_id", "Task"])
eeg_df["EEG"]=1

et_df = pd.DataFrame(et_subs_per_task_overall, columns=["bids_id", "Task"])
et_df["ET"] = 1
#et_df.drop(columns=["Task"], inplace=True)

# Add task-specific columns
for task in tasks:
    eeg_df[task] = eeg_df["Task"].apply(lambda x: 1 if x == task else 0)
eeg_df.drop(columns=["Task"], inplace=True)
   
for task in tasks:
    et_df[task] = et_df["Task"].apply(lambda x: 1 if x == task else 0)
et_df.drop(columns=["Task"], inplace=True)
   
    
    
discrepancies_df = et_df.merge(eeg_df, on="bids_id", how="outer")
discrepancies_df = discrepancies_df.drop_duplicates(subset=['bids_id'])
# Fill NaN with 0 (indicating incomplete stages)
discrepancies_df.fillna(0, inplace=True)
# Merge columns
for col in tasks:
    discrepancies_df[col] = discrepancies_df[[f"{col}_x", f"{col}_y"]].max(axis=1)  # Take the maximum value (1 if either is 1)
    discrepancies_df.drop(columns=[f"{col}_x", f"{col}_y"], inplace=True)  # Drop the original columns   
    
discrepancies_df

In [ ]:
discrepancies_df=discrepancies_df.loc[(discrepancies_df.ET==0) | (discrepancies_df.EEG==0)]

In [ ]:
discrepancies_df

In [ ]:
discrepancies_df.to_csv(f"../../tracking/output_dfs/missingness/{date}/participants__extracted_et_but_no_bids_eeg.csv", index=False)

### 4. Organize and export .edf, .asc and .txt files

In [ ]:
input_dir= f"../../derivatives/et_rt_analysis/"

In [ ]:
output_dir= f"../../derivatives/et_rt_analysis/derivatives/task_analysis/et_task_outputs/"

In [ ]:
file_types = ["edf", "txt", "asc"]
for task in tqdm.tqdm(tasks):
    # Collect all files for this task and all types
    file_list = []
    # .edf files
    if "edf" in file_types:
        edf_pattern = f"../../derivatives/et_rt_analysis/sub*/*{task}*.edf"
        file_list.extend(glob.glob(edf_pattern, recursive=True))  # Changed to extend
    
    # .txt files
    if "txt" in file_types:
        txt_pattern = f"../../derivatives/et_rt_analysis/sub*/*_{task}.txt"
        file_list.extend(glob.glob(txt_pattern, recursive=True))  # Changed to extend
    
    # .asc files
    if "asc" in file_types:
        asc_pattern = f"../../sourcedata/*/et/*/*_{task}.asc"
        file_list.extend(glob.glob(asc_pattern, recursive=True))  # Changed to extend

    # Copy files to destination
    for file in file_list:
        print("Processing file:", file)
        source_path = file
        source_path_id = source_path.split("/")[-1]
        
        # Choose subfolder based on file extension
        if file.endswith('.edf'):
            subfolder = "edf_files"
        elif file.endswith('.txt'):
            subfolder = "txt_files"
        elif file.endswith('.asc'):
            subfolder = "asc_files"
        else:
            continue
            
        destination_dir = f"{os.path.dirname(output_dir)}/{task}_files/{subfolder}/"
        os.makedirs(destination_dir, exist_ok=True)
        destination_path = os.path.join(destination_dir, source_path_id)
        shutil.copy(source_path, destination_path)
        
        if "missing_eeg" in source_path:
            continue